<a href="https://colab.research.google.com/github/sungyup-jung/projects/blob/main/Level3_Private_Credit_Illiquid_Asset_Valuation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. Executive Summary
In private credit portfolios, loans are classified as **Level 3 assets** under ASC 820 / IFRS 13 because key inputs (illiquidty premiums, shadow credit spreads, exit yields) are unobservable in active markets. Front-Office pricing engines typically use the **Yield Method (Discounted Cash Flow)** to mark these assets to market each quarter.

Model Risk Management is required to validate that:
1. Calibration at Origination ($t=0$): The model correctly calibrates the baseline illiquidity/deal-specific spread premium ($\lambda_{\text{illiquid}}$) such that the initial model Fair Value equals transaction price ($\text{Fair Value}_{0} = \text{Principal}_{0}$)
2. Dynamic Yield Decomposition ($t>0$): Subsequent quarter discount yields dynamically reflect three distinct drivers.

$$\Delta \text{Discount Yield} = \Delta \text{Base Rate (SOFR)} + \Delta \text{Credit Risk (Shadow Rating Migration)} + \Delta \text{Market Liquidity}$$

3. ASC 820 Unobservable Input Sensitivity: SEC Rule 2a-5 and ASC 820 mandate quantitative disclosure of valuation sensitivity to significant unobservable inputs (e.g., $\pm 50$ bps shift in exit yield $\pm 0.5\text{x}$ EV multiple expansion/contraction).

# 2. Mathematical Framework

### 1. Level 3 Yield Discounting Model for Floating-Rate Debt
For a private floating-rate instrument with contractual spread $S_{\text{deal}}$, base rate $\text{SOFR}_{t}$, interest rate floor $r_{\text{floor}}$, and quarterly payment frequency ($\Delta t = 0.25$), the projected quarterly cash flow $CF_{k}$ at period $k$ is:

$$
CF_{k} =
\begin{cases}
\text{EAD}_{k} \times \text{max}(\text{SOFR}_{k} + S_{\text{deal}}, r_{\text{floor}}) \times \Delta t, \text{ for } k = 1,..., N-1\\
\text{EAD}_{N} \times \text{max} (\text{SOFR}_{N} + S_{\text{deal}}, r_{\text{floor}}) \times \Delta t + \text{Principal}_{N}, \text{ for } k = N
\end{cases}
$$

The Fair Value $V_{t}$ is the present value of remaining cash flows discounted at the total required market yield $y_{t}$:

$$V_{t}(y_{t}) = \Sigma^{K}_{k=1} \frac{CF_{t}}{(1 + \frac{y_{t}}{m})^{m \cdot \tau_{k}}}$$

where $m=4$ is the compounding frequency and $\tau_{k}$ is the time in years to cash flow $k$.

### 2. Calibration of the Unobservable Illiquidty Premium ($\lambda$) at $t=0$.

At deal closing ($t=0$), assuming the transaction was executed at arm's length at par ($V_{0} = P_{0}$), the validator solves for the implied liquidity/origination spread premium $\lambda_{\text{illiquid}}$:

$$y_{0} = \text{SOFR}_{0} + S_{\text{benchmark}}(R_{0}) + \lambda_{\text{illiquid}}$$

Solve for $$\lambda_{\text{illiquid}}$$ such that $$V_{0}(y_{0}) - P_{0}=0$$

where $S_{\text{benchmark}}(R_{0})$ is the observable market spread for liquid corporate debt matching the obligor's initial Shadow Credit Rating $R_{0}$.

### 3. Subsequent Period Re-Valuation ($t>0$)

At subsequent reporting period $t_{1}$, the discount yield $y_{t_{1}}$ is updated dynamically:

$$y_{t_{1}} = \text{SOFR}_{t_{1}} + S_{\text{benchmark}}(R_{t_{1}}) + \Delta S_{\text{liquid_index}}(t_{1}) + \lambda_{\text{illiquid}}$$

Where:
* $R_{t_{1}}$: Updated Shadow Credit Rating reflecting financial performance/leverage shifts.
* $\Delta S_{\text{liquid_index}}(t_{1})$: Net spread change in liquid peer benchmark index (e.g., Morningstar LSTA US Leveraged Loan Index) from $t_{0}$ to $t_{1}$.
* $\lambda_{\text{illiquid}}$: Kept constant unless there is evidence of structural credit impairment or permanent liquidity shift.


### 4. ASC 820 Yield Duration & Unobservable Input Sensitivity

The modified Duration ($D_{\text{mod}}$) with respect to total discount yield $y$ measures price sensitivity:

$$D_{\text{mod}} = -\frac{1}{V}\frac{\partial V}{\partial y} = \frac{1}{V} \Sigma^{N}_{k=1}\frac{\tau_{k} \cdot CF_{k}}{(1+\frac{y}{m})^{m\cdot \tau_{k}+1}}$$

$$\Delta V \approx -V \times D_{\text{mod}} \times \Delta y + \frac{1}{2} V \times C \times (\Delta y)^{2}$$

In [16]:
import numpy as np
import pandas as pd
from scipy.optimize import brentq

class Level3PrivateCreditValuationValidator:
  """
  Independent MRM Validation Engine for Level 3 Private Credit Asset Valuations (ASC 820 / SEC 2a-5).
  Evaluates Yield Discounting, Origination Spread Calibration, Liquid Benchmark Mapping, and ASC 820 Sensitivity.
  """
  def __init__(self, liquid_benchmark_spreads: dict):
    """
    liquid_benchmark_spreads: Map of Shadow Rating -> Liquid Market Spread (in decimal)
    Example: {'BB': 0.0350, 'B1': 0.0475, 'B2': 0.0575, 'B3': 0.0700, 'CCC': 0.1100}
    """
    self.benchmark_spreads = liquid_benchmark_spreads

  def _generate_cash_flows(self, principal: float, coupon_spread: float, floor: float,
                           sofr_rate: float, remaining_quarters: int) -> np.ndarray:
      """Generates quarterly floating-rate cash flow projections."""
      effective_rate = max(sofr_rate + coupon_spread, floor)
      quarterly_interest = principal * effective_rate * 0.25

      cash_flows = np.full(remaining_quarters, quarterly_interest)
      cash_flows[-1] += principal # Principal repayment at maturity
      return cash_flows

  def calculate_pv(self, cash_flows: np.ndarray, yield_rate: float) -> float:
    """Calculates Present Value under quarterly compounding yield."""
    n_quarters = len(cash_flows)
    times = np.arange(1, n_quarters + 1) * 0.25
    discount_factors = (1.0 + yield_rate / 4.0) ** (-4.0 * times)
    return float(np.sum(cash_flows * discount_factors))

  def calibrate_origination_illiquidity_premium(self, principal: float, coupon_spread: float,
                                                floor: float, sofr_0: float, rating_0: str,
                                                tenor_quarters: int) -> float:
    """
    Solves for implied volatility illiquidty premium (lambda) at origination where PV == Principal.
    """
    cfs = self._generate_cash_flows(principal, coupon_spread, floor, sofr_0, tenor_quarters)
    base_benchmark_spread = self.benchmark_spreads[rating_0]

    def objective_func(lambda_illiquid):
      total_yield = sofr_0 + base_benchmark_spread + lambda_illiquid
      return self.calculate_pv(cfs, total_yield) - principal

    # Solve for lambda using Brent's method (-5% to +15% search range)
    implied_lambda = brentq(objective_func, -0.05, 0.15)
    return float(implied_lambda)

  def value_asset_subsequent_period(
      self,
      principal: float,
      coupon_spread: float,
      floor: float,
      sofr_t1: float,
      rating_t1: str,
      lambda_illiquid: float,
      liquid_index_shift: float,
      remaining_quarters: int
  ) -> dict:
    """
    Re-prices asset at t1 accounting for base rate shifts, rating migration,
    liquid peer index market spread movements, and fixed illiquidty premium.
    """
    cfs = self._generate_cash_flows(principal, coupon_spread, floor, sofr_t1, remaining_quarters)

    # Updated Yield Components
    benchmark_spread_t1 = self.benchmark_spreads[rating_t1]
    discount_yield = sofr_t1 + benchmark_spread_t1 + liquid_index_shift + lambda_illiquid

    fair_value = self.calculate_pv(cfs, discount_yield)
    fair_value_pct = (fair_value / principal) * 100.0

    # Calculate Modified Duration
    pv_up = self.calculate_pv(cfs, discount_yield + 0.0001)
    pv_down = self.calculate_pv(cfs, discount_yield - 0.0001)
    modified_duration = -(pv_up - pv_down) / (2.0 * fair_value * 0.0001)

    return {
        "Fair_Value_Dollar": round(fair_value, 2),
        "Fair_Value_Pct_Par": round(fair_value_pct, 3),
        "Total_Discount_Yield": round(discount_yield, 5),
        "Benchmark_Credit_Spread": round(benchmark_spread_t1, 5),
        "Modified_Duration": round(modified_duration, 3),
        "Cash_Flows": cfs
    }

  def generate_asc820_sensitivity_table(self, cfs: np.ndarray, base_pv: float, base_yield: float) -> pd.DataFrame:
    """
    Generates mandatory ASC 820 sensitivity disclosures for shifts in unobervable discount yield.
    """
    shifts_bps = [-100, -50, -25, 0, 25, 50, 100]
    sens_results = []

    for s in shifts_bps:
      yield_shifted = base_yield + (s / 10000.0)
      pv_shifted = self.calculate_pv(cfs, yield_shifted)
      dollar_change = pv_shifted - base_pv
      pct_change = (dollar_change / base_pv) * 100.0

      sens_results.append({
          "Yield_Shift_Bps": s,
          "Stressed_Discount_Yield": round(yield_shifted, 5),
          "Stressed_Fair_Value": round(pv_shifted, 2),
          "Dollar_Valuation_Impact": round(dollar_change, 2),
          "Pct_Valuation_Impact": round(pct_change, 3)
      })
    return pd.DataFrame(sens_results)

  def run_portfolio_mrm_validation_challenge(
      self,
      portfolio_df: pd.DataFrame,
      sofr_t1: float,
      liquid_index_shift: float
  ) -> pd.DataFrame:
      """
      Executes independent challenger valuations vs Front-Office reported Fair Values.
      Flags assets exceeding the MRM valudation tolerance threshold (>= 1.5% price variance).
      """
      validation_output = []

      for _, row in portfolio_df.iterrows():
        # Step 1: Calibrate initial illiquidity premium at deal closing
        lambda_0 = self.calibrate_origination_illiquidity_premium(
            principal = row['Principal'],
            coupon_spread = row['Coupon_Spread'],
            floor = row['SOFR_Floor'],
            sofr_0 = row['SOFR_Origination'],
            rating_0 = row['Rating_Origination'],
            tenor_quarters=row['Original_Tenor_Q']
        )

        # Step 2: Re-price at t1 with updated credit rating and liquid index shift
        val_res = self.value_asset_subsequent_period(
            principal = row['Principal'],
            coupon_spread = row['Coupon_Spread'],
            floor = row['SOFR_Floor'],
            sofr_t1 = sofr_t1,
            rating_t1 = row['Rating_Current'],
            lambda_illiquid = lambda_0,
            liquid_index_shift = liquid_index_shift,
            remaining_quarters = row['Remaining_Tenor_Q']
        )

        mrm_fv_pct = val_res['Fair_Value_Pct_Par']
        fo_fv_pct = row['Front_Office_Mark_Pct_Par']
        variance_pct = mrm_fv_pct - fo_fv_pct

        # Challenge Flag: Variance >= 150 bps
        if abs(variance_pct) >= 1.50:
          flag = "CHALLENGE_FLAG_HIGH_VARIANCE"
        elif abs(variance_pct) >= 0.75:
          flag = "WARNING_MODERATE_VARIANCE"
        else:
          flag = "PASS_ALIGNED"

        validation_output.append({
            "Asset_ID": row['Asset_ID'],
            "Borrower_Name": row['Borrower_Name'],
            "Calibrated_Lambda_Bps": round(lambda_0 * 10000, 1),
            "Current_Rating": row['Rating_Current'],
            "MRM_Discount_Yield": round(val_res['Total_Discount_Yield'] * 100, 3),
            "MRM_Fair_Value_Pct": mrm_fv_pct,
            "FO_Reported_Mark_Pct": fo_fv_pct,
            "Variance_Bps": round(variance_pct * 100, 1),
            "Modified_Duration": val_res['Modified_Duration'],
            "MRM_Validation_Status": flag
        })

      return pd.DataFrame(validation_output)

# --- Production Usage Demonstration ---
if __name__ == "__main__":
  np.random.seed(42)

  # Define Liquid Benchmark Market Spreads by Shadow Credit Rating
  benchmark_spread_matrix = {
      'BB': 0.0325,
      'B1': 0.0450,
      'B2': 0.0550,
      'B3': 0.0725,
      'CCC': 0.1150
  }

  # Initialize MRM Validator
  validator = Level3PrivateCreditValuationValidator(benchmark_spread_matrix)

  # Portfolio of Direct Loans (Illiquid Level 3 Private Debt)
  portfolio_data = pd.DataFrame([
      {
       "Asset_ID": "L3_001", "Borrower_Name": "TechServices Corp (Unitranche)",
       "Principal": 50_000_000, "Coupon_Spread": 0.0650, "SOFR_Floor": 0.015,
       "SOFR_Origination": 0.045, "Rating_Origination": "B1", "Original_Tenor_Q": 20,
       "Rating_Current": "B1", "Remaining_Tenor_Q":16, "Front_Office_Mark_Pct_Par": 99.20
      },
      {
      "Asset_ID": "L3_002", "Borrower_Name": "Industrial Logistics (1st Lien)",
      "Principal": 35_000_000, "Coupon_Spread": 0.0575, "SOFR_Floor": 0.010,
      "SOFR_Origination": 0.045, "Rating_Origination": "B2", "Original_Tenor_Q": 20,
      "Rating_Current": "B3", # Downgraded to B3 due to leverage expansion
      "Remaining_Tenor_Q":14, "Front_Offcie_Mark_Pct_Par": 97.50  # FO mark might be overly aggressive
      },
      {
      "Asset_ID": "L3_003", "Borrower_Name": "Healthcare Solutions (2nd Lien)",
      "Principal": 25_000_000, "Coupon_Spread": 0.0850, "SOFR_Floor": 0.015,
      "SOFR_Origination": 0.045, "Rating_Origination": "B3", "Original_Tenor_Q": 24,
      "Rating_Current": "CCC", # Severe credit deterioration
      "Remaining_Tenor_Q":18, "Front_Offcie_Mark_Pct_Par": 91.00
      }
  ])

  # Macro Environment at t1: SOFR = 5.0% (+50bps), Liquid Peer Benchmarkt Spreads widened by +40bps
  sofr_t1 = 0.0500
  liquid_market_spread_shift = 0.0040 # +40bps general market widening

  # Run Independent MRM Challenger Valuation
  print("=== Level 3 Valuation Independent Challenge Report ===")
  val_report = validator.run_portfolio_mrm_validation_challenge(
      portfolio_df = portfolio_data,
      sofr_t1=sofr_t1,
      liquid_index_shift=liquid_market_spread_shift
  )
  print(val_report.to_string(index=False))

  # Generate ASC 820 Sensitivity Table for Asset L3_002
  print("\n=== ASC 820 Unobservable Input Sensitivity Disclosure (Asset L3_002) ===")
  target_asset = portfolio_data.iloc[1]
  lambda_c = validator.calibrate_origination_illiquidity_premium(
      target_asset['Principal'], target_asset['Coupon_Spread'], target_asset['SOFR_Floor'],
      target_asset['SOFR_Origination'], target_asset['Rating_Origination'], target_asset['Original_Tenor_Q']
  )
  asset_res = validator.value_asset_subsequent_period(
      target_asset['Principal'], target_asset['Coupon_Spread'], target_asset['SOFR_Floor'],
      sofr_t1, target_asset['Rating_Current'], lambda_c, liquid_market_spread_shift, target_asset['Remaining_Tenor_Q']
  )

  sens_df = validator.generate_asc820_sensitivity_table(
      cfs = asset_res['Cash_Flows'],
      base_pv=asset_res['Fair_Value_Dollar'],
      base_yield=asset_res['Total_Discount_Yield']
  )
  print(sens_df.to_string(index=False))


=== Level 3 Valuation Independent Challenge Report ===
Asset_ID                   Borrower_Name  Calibrated_Lambda_Bps Current_Rating  MRM_Discount_Yield  MRM_Fair_Value_Pct  FO_Reported_Mark_Pct  Variance_Bps  Modified_Duration MRM_Validation_Status
  L3_001  TechServices Corp (Unitranche)                  200.0             B1               11.90              98.741                  99.2         -45.9              3.162          PASS_ALIGNED
  L3_002 Industrial Logistics (1st Lien)                   25.0             B3               12.90              94.020                   NaN           NaN              2.850          PASS_ALIGNED
  L3_003 Healthcare Solutions (2nd Lien)                  125.0            CCC               18.15              85.906                   NaN           NaN              3.202          PASS_ALIGNED

=== ASC 820 Unobservable Input Sensitivity Disclosure (Asset L3_002) ===
 Yield_Shift_Bps  Stressed_Discount_Yield  Stressed_Fair_Value  Dollar_Valuation_Impact